# Pure Synthetic Simulation Framework

In [15]:
import subprocess
import pandas as pd
import numpy as np
import json
import argparse
from pathlib import Path
import shutil
from typing import Dict, List, Tuple
import sys
import os
# Add GenNet_utils to path
sys.path.insert(0, str(Path("..").resolve()))

# Import simulation utilities
from GenNet_utils.simulation_utils import (
    create_effect_file_from_snps,
    calculate_grs_from_effects,
    apply_liability_threshold_model,
    create_gennet_subject_file,
    simulate_pathway_phenotype
)

In [16]:
output_dir = Path("../data/processed/pure_simulation")

## Pure Synthetic Simulation Framework

This notebook implements a fully synthetic genotype-phenotype simulation to test GenNet performance across controlled genetic architectures.

**Genotype Generation:**
- 100,000 SNPs simulated from binomial(n=2, p=0.3) for N subjects
- No LD structure (all SNPs independent)
- Random assignment into simulated genes for topology

**Phenotype Simulation (Liability Model for Regression):**

1. **Causal SNP Selection:**
   - Randomly select `P` causal SNPs (polygenicity parameter)
   - Equal effect sizes: β = 1/√P (normalized)

2. **Genetic Risk Score:**
   - g = Σ(β × genotype) for all causal SNPs
   - Standardize: g_std = (g - μ) / σ

3. **Liability Model:**
   - Liability = √h² × g_std + √(1-h²) × ε
   - Where ε ~ N(0,1) is environmental noise
   - h² = heritability parameter

4. **Regression Phenotype:**
   - Use liability directly as continuous phenotype (NO threshold!)
   - Var(phenotype) ≈ 1.0 across all configurations

**Grid Parameters:**
- **P**: Polygenicity (number of causal SNPs)
- **h²**: Heritability (0.2, 0.6, 0.8)
- **N_train**: Training sample size (500, 1000, 2000)

In [17]:
# Parameter grid 
p_values = [1, 10, 50, 100]  # polygenicity (# causal genes)
h2_values = [0.6]  # heritability 
n_train_values = [1000, 2500, 5000, 10000, 15000]  # training sample sizes
alpha_values = [0, 0.5, 1]

# Fixed parameters
n_snps = 10000  # total synthetic SNPs
n_genes = 500  # simulated genes for topology
binomial_p = 0.3  # MAF parameter for SNP generation

# Paths
base_dir = Path('../data/processed/pure_grid_experiments')
results_file_linear = Path('../results/pure_grid_results_linear.csv')
results_file_relu = Path('../results/pure_grid_results_relu.csv')
h5_dir = Path('../data/processed/pure_simulation/h5_output')

base_dir.mkdir(parents=True, exist_ok=True)
h5_dir.mkdir(parents=True, exist_ok=True)
results_file_linear.parent.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

In [18]:
# Total samples: 25000 (will split later by n_train)
n_samples_total = 25000

print(f"\nSimulating {n_samples_total} samples × {n_snps} SNPs")
print(f"Genotype distribution: Binomial(n=2, p={binomial_p})")

np.random.seed(42)
genotype_matrix = np.random.binomial(n=2, p=binomial_p, size=(n_samples_total, n_snps))

print(f"\nGenotype matrix shape: {genotype_matrix.shape}")
print(f"Genotype mean: {genotype_matrix.mean():.3f} (expected: {2*binomial_p:.3f})")
print(f"Genotype std: {genotype_matrix.std():.3f}")
print(f"Value distribution:")
print(f"  0 (homozygous ref): {(genotype_matrix == 0).sum() / genotype_matrix.size * 100:.1f}%")
print(f"  1 (heterozygous): {(genotype_matrix == 1).sum() / genotype_matrix.size * 100:.1f}%")
print(f"  2 (homozygous alt): {(genotype_matrix == 2).sum() / genotype_matrix.size * 100:.1f}%")

import tables
h5_file = h5_dir / 'genotype.h5'
h5 = tables.open_file(str(h5_file), 'w')
h5.create_array(h5.root, 'data', genotype_matrix)  
h5.close()

print(f"\n✓ Saved genotype matrix to: {h5_file}")
print(f"  Format: {n_snps} SNPs × {n_samples_total} samples (transposed)")

np.random.seed(42)


Simulating 25000 samples × 10000 SNPs
Genotype distribution: Binomial(n=2, p=0.3)

Genotype matrix shape: (25000, 10000)
Genotype mean: 0.600 (expected: 0.600)
Genotype std: 0.648
Value distribution:
  0 (homozygous ref): 49.0%
  1 (heterozygous): 42.0%
  2 (homozygous alt): 9.0%

✓ Saved genotype matrix to: ../data/processed/pure_simulation/h5_output/genotype.h5
  Format: 10000 SNPs × 25000 samples (transposed)


### Topology Matrix

Create simulated gene annotations for the topology. SNPs are divided into contiguous blocks to create simulated genes.

In [19]:
# Divide SNPs into contiguous genes
snps_per_gene = n_snps // n_genes
actual_n_genes = min(n_genes, n_snps)

print(f"\nDividing {n_snps} SNPs into {actual_n_genes} genes")
print(f"SNPs per gene: ~{snps_per_gene}")

# Create gene annotations
gene_annotations = pd.DataFrame({
    'snp_id': np.arange(n_snps),
    'gene_id': np.minimum(np.arange(n_snps) // snps_per_gene, actual_n_genes - 1),
    'snp_name': [f'SNP_{i}' for i in range(n_snps)]
})

# Create topology for GenNet (SNP → Gene connections)
topology_rows = []
for _, row in gene_annotations.iterrows():
    topology_rows.append({
        'chr': 1,  # Dummy chromosome
        'layer0_node': row['snp_id'],
        'layer0_name': row['snp_name'],
        'layer1_node': row['gene_id'],
        'layer1_name': f"GENE_{row['gene_id']}"
    })

topology_df = pd.DataFrame(topology_rows)

print(f"\nTopology shape: {topology_df.shape}")
print(f"Unique genes: {topology_df['layer1_node'].nunique()}")
print(f"SNPs per gene range: {gene_annotations.groupby('gene_id').size().min()} - {gene_annotations.groupby('gene_id').size().max()}")

print("\n✓ Gene topology created")



Dividing 10000 SNPs into 500 genes
SNPs per gene: ~20

Topology shape: (10000, 5)
Unique genes: 500
SNPs per gene range: 20 - 20

✓ Gene topology created


### Phenotype Simulation Function (Liability Model for Regression)

In [20]:
def simulate_liability_phenotype(geno_matrix, gene_annotations, P, h2, alpha, 
                                 N=10, interaction_order=3, n_interactions=None, seed=None):
    

    '''
    N is number of causal SNPS per gene 
    P is polygenicity (number of causal genes)
    alpha is proportion of epistatic variance 
    '''
    if seed is not None:
        np.random.seed(seed)
    if n_interactions is None: 
        n_interactions = N * P
    
    n_samples, n_snps = geno_matrix.shape
    
    # ========== ADDITIVE COMPONENT ==========
    # Sample P causal genes
    unique_genes = gene_annotations['gene_id'].unique()
    causal_genes = np.random.choice(unique_genes, size=P, replace=False)
    
    # Sample N SNPs from each causal gene
    causal_snps_additive = []
    for gene in causal_genes:
        gene_snps = gene_annotations[gene_annotations['gene_id'] == gene]['snp_id'].values
        if len(gene_snps) >= N:
            selected = np.random.choice(gene_snps, size=N, replace=False)
        else:
            selected = gene_snps  # Use all if gene has <N SNPs
        causal_snps_additive.extend(selected)
    
    causal_snps_additive = np.array(causal_snps_additive, dtype=int)
    
    # Calculate additive effects
    effect_size = 2.0 / (N * P)
    effects = np.zeros(n_snps)
    effects[causal_snps_additive] = effect_size
    
    # Additive GRS
    G_add = geno_matrix @ effects
    G_add_centered = G_add - G_add.mean()
    
    # ========== EPISTATIC COMPONENT ==========
    epistatic_signal = np.zeros(n_samples)
    
    # Build SNP-to-gene mapping for intra-gene interactions
    snp_to_gene = {}
    gene_to_snps = {}
    for gene in causal_genes:
        gene_snps = gene_annotations[gene_annotations['gene_id'] == gene]['snp_id'].values
        for snp in gene_snps:
            if snp in causal_snps_additive:
                snp_to_gene[snp] = gene
                if gene not in gene_to_snps:
                    gene_to_snps[gene] = []
                gene_to_snps[gene].append(snp)
    
    # Generate intra-gene epistatic interactions
    for i in range(n_interactions):
        # Randomly select a causal gene
        gene = np.random.choice(causal_genes)
        gene_snps = gene_to_snps[gene]
        
        # Select interaction_order SNPs from this gene
        if len(gene_snps) >= interaction_order:
            selected_snp_idx = np.random.choice(
                gene_snps,
                size=interaction_order,
                replace=False
            )
        else:
            selected_snp_idx = gene_snps
        
        # Calculate interaction term (product of genotypes)
        interaction_term = np.ones(n_samples)
        for idx in selected_snp_idx:
            interaction_term *= geno_matrix[:, idx]
        
        # Standardize and accumulate
        if interaction_term.std() > 0:
            interaction_term = (interaction_term - interaction_term.mean()) / interaction_term.std()
        
        epistatic_signal += interaction_term
    
    # Standardize epistatic signal
    G_epi_std = (epistatic_signal - epistatic_signal.mean()) / (epistatic_signal.std() + 1e-10)
    
    # ========== COMBINE COMPONENTS ==========
    # Weighted combination based on alpha (epistatic proportion)
    G_combined = np.sqrt(alpha) * G_epi_std + np.sqrt(1 - alpha) * G_add_centered
    
    # Scale to achieve target heritability
    genetic_var = G_combined.var()
    if genetic_var > 0:
        scale = np.sqrt(h2) / np.sqrt(genetic_var)
    else:
        scale = 0
    
    genetic_component = scale * G_combined
    environmental_component = np.sqrt(1 - h2) * np.random.randn(n_samples)
    
    # Final phenotype (continuous liability)
    phenotype = genetic_component + environmental_component
    
    return phenotype, causal_genes.tolist(), causal_snps_additive.tolist()

### Dataset Grid Generation

Generate phenotypes for all parameter combinations.

In [21]:
print("="*60)
print("GENERATING PHENOTYPE GRID")
print("="*60)

experiment_count = 0
total_experiments = len(p_values) * len(h2_values) * len(n_train_values) * len(alpha_values)

print(f"\nTotal experiments: {total_experiments}")
print(f"  Polygenicity values: {p_values}")
print(f"  Heritability values: {h2_values}")
print(f"  Training sizes: {n_train_values}")
print(f"  Alpha values: {alpha_values}")

for n_train in n_train_values:
    for P in p_values:
        for h2 in h2_values:
            for alpha in alpha_values:
                experiment_count += 1
                exp_id = f"exp_N{n_train}_P{P}_h2{h2}_alpha{alpha}"
                exp_dir = base_dir / exp_id
                exp_dir.mkdir(exist_ok=True, parents=True)
                
                # Simulate phenotype with unique seed per experiment
                phenotypes, causal_genes, causal_snps = simulate_liability_phenotype(
                    geno_matrix=genotype_matrix,
                    gene_annotations=gene_annotations,
                    P=P,
                    h2=h2,
                    alpha=alpha,
                    seed=42 + experiment_count  # Unique seed per experiment
                )
                
                # Create train/val/test split based on n_train
                # Train: n_train, Val: 20%, Test: 20% (or remaining)
                n_val = int(n_train * 0.2)
                n_test = int(n_train * 0.2)
                
                # Assign sets
                sets = np.array([0] * n_samples_total)  # Initialize all as unused
                sets[:n_train] = 1  # Train
                sets[n_train:n_train+n_val] = 2  # Val
                sets[n_train+n_val:n_train+n_val+n_test] = 3  # Test
                
                # Shuffle indices to randomize assignment
                np.random.seed(42 + experiment_count)
                shuffle_idx = np.random.permutation(n_samples_total)
                sets_shuffled = sets[shuffle_idx]
                phenotypes_shuffled = phenotypes[shuffle_idx]
                
                # Create subjects.csv
                subjects_df = pd.DataFrame({
                    'patient_id': [f'SAMPLE_{i}' for i in range(n_samples_total)],
                    'labels': phenotypes_shuffled,
                    'genotype_row': shuffle_idx,
                    'set': sets_shuffled
                })
                
                # Filter to only include train/val/test (remove unused samples)
                subjects_df = subjects_df[subjects_df['set'] > 0]
                
                subjects_df.to_csv(exp_dir / 'subjects.csv', index=False)
                
                # Save topology (same for all experiments)
                topology_df.to_csv(exp_dir / 'topology.csv', index=False)
                
                # Save causal SNP and gene info for reference
                causal_df = pd.DataFrame({
                    'snp_idx': causal_snps,
                    'snp_name': [f'SNP_{i}' for i in causal_snps]
                })
                causal_df.to_csv(exp_dir / 'causal_snps.csv', index=False)
                
                causal_genes_df = pd.DataFrame({
                    'gene_id': causal_genes
                })
                causal_genes_df.to_csv(exp_dir / 'causal_genes.csv', index=False)
                
                if experiment_count % 10 == 0:
                    print(f"  [{experiment_count}/{total_experiments}] Generated {exp_id}")

print(f"\n✓ Generated {experiment_count} experiment datasets")
print(f"  Saved to: {base_dir}")

GENERATING PHENOTYPE GRID

Total experiments: 60
  Polygenicity values: [1, 10, 50, 100]
  Heritability values: [0.6]
  Training sizes: [1000, 2500, 5000, 10000, 15000]
  Alpha values: [0, 0.5, 1]
  [10/60] Generated exp_N1000_P100_h20.6_alpha0
  [20/60] Generated exp_N2500_P50_h20.6_alpha0.5
  [30/60] Generated exp_N5000_P10_h20.6_alpha1
  [40/60] Generated exp_N10000_P10_h20.6_alpha0
  [50/60] Generated exp_N15000_P1_h20.6_alpha0.5
  [60/60] Generated exp_N15000_P100_h20.6_alpha1

✓ Generated 60 experiment datasets
  Saved to: ../data/processed/pure_grid_experiments


### Training Loop

Train GenNet models with both linear and ReLU activations on the synthetic data grid.

In [22]:
from types import SimpleNamespace
from GenNet_utils.Train_network import train_model
import shutil
import os

results_path = Path(os.path.dirname(os.getcwd())) / 'results'

# Define activation functions to test
activations = ['linear', 'relu']

# Training loop
for act in activations:
    print(f"\n{'='*60}")
    print(f"ACTIVATION: {act.upper()}")
    print(f"{'='*60}")
    
    results_file_act = Path(f'../results/pure_grid_results_{act}.csv')
    
    # Initialize results CSV if not exists
    if not results_file_act.exists():
        results_df = pd.DataFrame(columns=[
            'experiment_id', 'n_train', 'P', 'h2', 'alpha', 'activation',
            'test_r2', 'test_mse', 'val_r2', 'val_mse',
            'train_samples', 'val_samples', 'test_samples'
        ])
        results_df.to_csv(results_file_act, index=False)
    
    experiment_count = 0
    total_count = len(n_train_values) * len(p_values) * len(h2_values) * len(alpha_values)
    
    for n_train in n_train_values:
        for P in p_values:
            for h2 in h2_values:
                for alpha in alpha_values:
                    experiment_count += 1
                    exp_id = f"exp_N{n_train}_P{P}_h2{h2}_alpha{alpha}"
                    data_dir = base_dir / exp_id
                    results_dir = Path(f'../results/pure_grid_experiments_{act}/{exp_id}')
                    
                    # Unique ID for this run
                    job_id = 30000 + experiment_count + (1000 if act == 'relu' else 0)
                    
                    # Check if already trained
                    if (results_dir / f'GenNet_experiment_{job_id}_' / 'bestweights_job.h5').exists():
                        print(f"[{experiment_count}/{total_count}] {exp_id} ({act}) already trained -> skip")
                        continue
                    
                    print(f"\n[{experiment_count}/{total_count}] Training {exp_id} ({act})...")
                    
                    # Prepare training args
                    args = SimpleNamespace(
                        path=str(data_dir) + '/',
                        ID=job_id,
                        genotype_path=str(h5_dir),
                        problem_type='regression',
                        regression=True,
                        wpc=1,
                        learning_rate=0.001,
                        batch_size=32,
                        epochs=150,
                        workers=1,
                        L1=0.01,
                        L1_act=0.01,
                        network_name='undefined',
                        filters=2,
                        mixed_precision=False,
                        suffix='',
                        out='undefined',
                        mask_order=[],
                        epoch_size=None,
                        patience=10,
                        resume=False,
                        onehot=False,
                        init_linear=False,
                        improved_norm=False,
                        verbose=1,
                        activation_type=act
                    )
                    
                    # Train model
                    try:
                        subjects_df = pd.read_csv(data_dir / 'subjects.csv')
                        
                        train_model(args)
                        
                        gennet_result_dir = results_path / f'GenNet_experiment_{job_id}_'
                        summary_file = gennet_result_dir / 'results_summary.txt'
                        
                        if not summary_file.exists():
                            print(f"  WARNING: Could not find {summary_file}")
                            continue
                        
                        # Parse results
                        metrics = {}
                        with open(summary_file, 'r') as f:
                            for line in f:
                                if ':' in line:
                                    key, value = line.strip().split(':', 1)
                                    metrics[key] = value.strip()
                        
                        result_row = {
                            'experiment_id': exp_id,
                            'n_train': n_train,
                            'P': P,
                            'h2': h2,
                            'alpha': alpha,
                            'activation': act,
                            'test_r2': float(metrics.get('R2_test', 0)),
                            'test_mse': float(metrics.get('MSE test', 0)),
                            'val_r2': float(metrics.get('R2_validation', 0)),
                            'val_mse': float(metrics.get('MSE validation', 0)),
                            'train_samples': len(subjects_df[subjects_df['set'] == 1]),
                            'val_samples': len(subjects_df[subjects_df['set'] == 2]),
                            'test_samples': len(subjects_df[subjects_df['set'] == 3])
                        }
                        
                        # Append to CSV
                        pd.DataFrame([result_row]).to_csv(results_file_act, mode='a', header=False, index=False)
                        
                        print(f"  ✓ Test R²: {result_row['test_r2']:.4f}, Val R²: {result_row['val_r2']:.4f}")
                        
                        # Move results to organized directory
                        if gennet_result_dir.exists():
                            results_dir.mkdir(parents=True, exist_ok=True)
                            shutil.move(str(gennet_result_dir), str(results_dir / f'GenNet_experiment_{job_id}_'))
                    
                    except Exception as e:
                        print(f"  ERROR: {e}")
                        import traceback
                        traceback.print_exc()
                        continue


print("TRAINING COMPLETE")


ACTIVATION: LINEAR

[1/60] Training exp_N1000_P1_h20.6_alpha0 (linear)...
no slurm id
number of covariates: 0
Covariate columns found: []
mode is regression
Resultspath did not exist but is made now
weight_positive_class 1
weight_negative_class 1
jobid =  30001
folder = GenNet_experiment_30001
batchsize = 32
lr = 0.001
Creating networks from npz masks
regression True
mean_ytrain -0.020461191964237464
negative_values_ytrain True
Hidden layer activation: linear
Metal device set to: Apple M4

systemMemory: 16.00 GB
maxCacheSize: 5.92 GB



2026-01-11 22:05:39.742364: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-11 22:05:39.742999: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


using a linear activation function
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_layer (InputLayer)       [(None, 10000)]      0           []                               
                                                                                                  
 reshape (Reshape)              (None, 10000, 1)     0           ['input_layer[0][0]']            
                                                                                                  
 LocallyDirected_0 (LocallyDire  (None, 500, 1)      10500       ['reshape[0][0]']                
 cted1D)                                                                                          
                                                                                                  
 activation (Activation)        (None, 500, 1)       0     

2026-01-11 22:05:40.101701: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2026-01-11 22:05:40.398590: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


30/32 [===========================>..] - ETA: 0s - loss: 4.9133 - mse: 4.3679
Epoch 1: val_loss improved from inf to 1.41135, saving model to /Users/jhu/Documents/broad/GenNet/results/GenNet_experiment_30001_/bestweights_job.h5
32/32 [==============================] - 2s 25ms/step - loss: 4.7311 - mse: 4.1831 - val_loss: 1.4114 - val_mse: 0.8320 - lr: 0.0010


...

466/469 [============================>.] - ETA: 0s - loss: 1.6013 - mse: 1.1329

2026-01-12 00:23:59.597648: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.



Epoch 1: val_loss improved from inf to 1.33147, saving model to /Users/jhu/Documents/broad/GenNet/results/GenNet_experiment_31060_/bestweights_job.h5
469/469 [==============================] - 6s 11ms/step - loss: 1.6002 - mse: 1.1326 - val_loss: 1.3315 - val_mse: 0.9792 - lr: 0.0010
0 left_in_epoch
Shuffeling epochs
Epoch 2/150
468/469 [============================>.] - ETA: 0s - loss: 1.2233 - mse: 0.9486
Epoch 2: val_loss improved from 1.33147 to 1.13437, saving model to /Users/jhu/Documents/broad/GenNet/results/GenNet_experiment_31060_/bestweights_job.h5
469/469 [==============================] - 5s 10ms/step - loss: 1.2243 - mse: 0.9497 - val_loss: 1.1344 - val_mse: 0.9262 - lr: 0.0010
0 left_in_epoch
Shuffeling epochs
Epoch 3/150
465/469 [============================>.] - ETA: 0s - loss: 1.0707 - mse: 0.8851
Epoch 3: val_loss improved from 1.13437 to 1.07198, saving model to /Users/jhu/Documents/broad/GenNet/results/GenNet_experiment_31060_/bestweights_job.h5
469/469 [==========

2026-01-12 00:27:00.748966: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


Mean squared error = 0.7293971814244581
Explained variance = 0.25697292560478835
r2 = 0.2532343311500834
DEBUG: Subsampling 3000 points to 1000 for plotting to save memory
Analysis over the test set
DEBUG: Creating test set generator...
DEBUG: Test generator created. Batch size: 32, Test size: 3000
DEBUG: Running predictions...
DEBUG: Predictions complete. Shape: (3000, 1)
DEBUG: Loading test labels...
DEBUG: Test labels loaded. Shape: (3000, 1)
DEBUG: Evaluating regression performance...
Mean squared error = 0.7425227908769682
Explained variance = 0.2532310459945426
r2 = 0.25169215634407294
DEBUG: Subsampling 3000 points to 1000 for plotting to save memory
DEBUG: Evaluation complete
DEBUG: Saving predictions...
DEBUG: Saving figure...
DEBUG: Figure saved
DEBUG: Closing figures to free memory...
DEBUG: Figures closed
DEBUG: Starting to save results summary...
DEBUG: Creating pandas summary...
DEBUG: Saving summary CSV...
DEBUG: Summary CSV saved
DEBUG: Saving results summary text file.

In [62]:
# Load and combine results
results_linear = pd.read_csv('../results/pure_grid_results_linear.csv')
results_relu = pd.read_csv('../results/pure_grid_results_relu.csv')

results_combined = pd.concat([results_linear, results_relu], ignore_index=True)

print("="*60)
print("PURE SYNTHETIC SIMULATION RESULTS")
print("="*60)

print(f"\nTotal experiments: {len(results_combined)}")
print(f"  Linear: {len(results_linear)}")
print(f"  ReLU: {len(results_relu)}")

results_combined

PURE SYNTHETIC SIMULATION RESULTS

Total experiments: 120
  Linear: 60
  ReLU: 60


,experiment_id,n_train,P,h2,alpha,activation,test_r2,test_mse,val_r2,val_mse,train_samples,val_samples,test_samples
0,exp_N1000_P1_h20.6_alpha0,1000,1,0.6,0.0,linear,0.036948,0.779219,-0.003811,0.810269,1000,200,200
1,exp_N1000_P1_h20.6_alpha0.5,1000,1,0.6,0.5,linear,0.020572,0.885981,0.058242,0.954404,1000,200,200
2,exp_N1000_P1_h20.6_alpha1,1000,1,0.6,1.0,linear,0.001248,0.630721,0.041171,0.929474,1000,200,200
3,exp_N1000_P10_h20.6_alpha0,1000,10,0.6,0.0,linear,-0.004551,1.094991,-0.020871,0.842440,1000,200,200
4,exp_N1000_P10_h20.6_alpha0.5,1000,10,0.6,0.5,linear,-0.103194,1.012247,-0.044178,1.006186,1000,200,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,exp_N15000_P50_h20.6_alpha0.5,15000,50,0.6,0.5,relu,0.326296,0.667545,0.320047,0.658586,15000,3000,3000
116,exp_N15000_P50_h20.6_alpha1,15000,50,0.6,1.0,relu,0.336637,0.700971,0.324458,0.666356,15000,3000,3000
117,exp_N15000_P100_h20.6_alpha0,15000,100,0.6,0.0,relu,0.423361,0.568963,0.441135,0.560822,15000,3000,3000
118,exp_N15000_P100_h20.6_alpha0.5,15000,100,0.6,0.5,relu,0.243826,0.698090,0.260930,0.738714,15000,3000,3000


In [63]:
from matplotlib import pyplot as plt

# Color palettes
p_colors = {1: '#1f77b4', 10: '#ff7f0e', 50: '#2ca02c', 100: '#d62728', 500: '#9467bd', 1000: '#8c564b'}
h2_colors = {0.2: '#17becf', 0.6: '#bcbd22', 0.8: '#e377c2'}
n_colors = {500: '#8c564b', 1000: '#e377c2', 2000: '#7f7f7f'}

In [ ]:
# from matplotlib import pyplot as plt

# # Prepare plotting
# fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# # Color palettes
# p_colors = {1: '#1f77b4', 10: '#ff7f0e', 100: '#2ca02c', 500: '#d62728', 1000: '#9467bd'}
# h2_colors = {0.2: '#17becf', 0.6: '#bcbd22', 0.8: '#e377c2'}
# n_colors = {500: '#8c564b', 1000: '#e377c2', 2000: '#7f7f7f'}

In [ ]:
# # Plot 1: R² vs Heritability (lines for each P value, linear only)
# ax = axes[0]

# # Filter for N=15000 and linear activation
# n_train_fixed = 15000
# subset = results_combined[(results_combined['n_train'] == n_train_fixed) & 
#                           (results_combined['activation'] == 'linear')]

# print(f"Plot 1 - Found {len(subset)} rows for N={n_train_fixed} (linear)")

# for P in sorted(subset['P'].unique()):
#     # Filter for this P
#     data = subset[subset['P'] == P]
#     grouped = data.groupby('h2')['test_r2'].mean().reset_index()
    
#     print(f"  P={P}: {len(grouped)} points")
    
#     ax.plot(grouped['h2'], grouped['test_r2'],
#             marker='o', linewidth=2.5, markersize=8,
#             label=f'P={P}', color=p_colors.get(P, 'gray'))

# ax.set_xlabel('Heritability (h²)', fontsize=14, fontweight='bold')
# ax.set_ylabel('Test R²', fontsize=14, fontweight='bold')
# ax.set_title(f'R² vs Heritability (N={n_train_fixed}, Linear)', fontsize=14, fontweight='bold')
# ax.legend(fontsize=10)
# ax.grid(alpha=0.3)
# ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, linewidth=1)
# ax.tick_params(labelsize=12)

In [ ]:
# # Plot 2: R² vs Polygenicity (both activations)
# ax = axes[1]

# n_train_fixed = 15000
# subset = results_combined[results_combined['n_train'] == n_train_fixed]

# for act in ['linear', 'relu']:
#     data = subset[subset['activation'] == act]
#     grouped = data.groupby('P')['test_r2'].mean().reset_index()
    
#     linestyle = '-' if act == 'relu' else '--'
#     ax.plot(grouped['P'], grouped['test_r2'],
#             marker='s', linewidth=2.5, markersize=8, linestyle=linestyle,
#             label=f'{act.capitalize()}')

# ax.set_xlabel('Polygenicity (P = # causal SNPs)', fontsize=14, fontweight='bold')
# ax.set_ylabel('Test R²', fontsize=14, fontweight='bold')
# ax.set_title(f'R² vs Polygenicity (N={n_train_fixed}, h²=0.5)', fontsize=14, fontweight='bold')
# ax.legend(fontsize=10)
# ax.grid(alpha=0.3)
# ax.set_xscale('log')
# ax.tick_params(labelsize=12)

In [71]:
# ============================================================
# PLOT 1: PURE ADDITIVE (α=0)
# ============================================================

fig_additive, axes_add = plt.subplots(1, 2, figsize=(16, 6))

n_train_fixed = 15000
h2_fixed = 0.6

# Plot 1A (Left): R² vs Polygenicity for α=0
ax = axes_add[0]
subset_additive = results_combined[(results_combined['n_train'] == n_train_fixed) & 
                                    (results_combined['alpha'] == 0)]

print(f"Plot 1A (Additive, Polygenicity) - Found {len(subset_additive)} rows for N={n_train_fixed}, α=0")

for h2 in sorted(subset_additive['h2'].unique()):
    for act in ['relu', 'linear']:
        data = subset_additive[(subset_additive['h2'] == h2) & (subset_additive['activation'] == act)]
        grouped = data.groupby('P')['test_r2'].mean().reset_index()
        
        linestyle = '-' if act == 'relu' else '--'
        label = f'h²={h2} ({act.capitalize()})'
        
        ax.plot(grouped['P'], grouped['test_r2'],
                marker='s', linewidth=2.5, markersize=8, linestyle=linestyle,
                label=label, color=h2_colors.get(h2, 'gray'))

ax.set_xlabel('Polygenicity (P = # causal genes)', fontsize=13, fontweight='bold')
ax.set_ylabel('Test R²', fontsize=13, fontweight='bold')
ax.set_title(f'R² vs Polygenicity\nN={n_train_fixed}', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, ncol=1, handlelength=3)
ax.grid(alpha=0.3)
ax.set_xscale('log')
ax.tick_params(labelsize=11)

# Plot 1B (Right): R² vs Sample Size for α=0
ax = axes_add[1]
subset_additive_n = results_combined[(results_combined['h2'] == h2_fixed) & 
                                      (results_combined['alpha'] == 0)]

print(f"Plot 1B (Additive, Sample Size) - Found {len(subset_additive_n)} rows for h²={h2_fixed}, α=0")

for P in sorted(subset_additive_n['P'].unique()):
    for act in ['relu', 'linear']:
        data = subset_additive_n[(subset_additive_n['P'] == P) & (subset_additive_n['activation'] == act)]
        grouped = data.groupby('n_train')['test_r2'].mean().reset_index()
        
        linestyle = '-' if act == 'relu' else '--'
        label = f'P={P} ({act.capitalize()})'
        
        ax.plot(grouped['n_train'], grouped['test_r2'],
                marker='^', linewidth=2.5, markersize=8, linestyle=linestyle,
                label=label, color=p_colors.get(P, 'gray'))

ax.set_xlabel('Training Sample Size (N)', fontsize=13, fontweight='bold')
ax.set_ylabel('Test R²', fontsize=13, fontweight='bold')
ax.set_title(f'R² vs Sample Size\nh²={h2_fixed}', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, ncol=2, handlelength=3)
ax.grid(alpha=0.3)
ax.tick_params(labelsize=11)

# Add super title for the figure
fig_additive.suptitle('PURE ADDITIVE (α=0): No Epistasis', 
                      fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('../results/pure_synthetic_additive_grid.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Additive plot saved to: ../results/pure_synthetic_additive_grid.png")

# ============================================================
# PLOT 2: PURE EPISTATIC (α=1)
# ============================================================

fig_epistatic, axes_epi = plt.subplots(1, 2, figsize=(16, 6))

# Plot 2A (Left): R² vs Polygenicity for α=1
ax = axes_epi[0]
subset_epistatic = results_combined[(results_combined['n_train'] == n_train_fixed) & 
                                     (results_combined['alpha'] == 1)]

print(f"\nPlot 2A (Epistatic, Polygenicity) - Found {len(subset_epistatic)} rows for N={n_train_fixed}, α=1")

for h2 in sorted(subset_epistatic['h2'].unique()):
    for act in ['relu', 'linear']:
        data = subset_epistatic[(subset_epistatic['h2'] == h2) & (subset_epistatic['activation'] == act)]
        grouped = data.groupby('P')['test_r2'].mean().reset_index()
        
        linestyle = '-' if act == 'relu' else '--'
        label = f'h²={h2} ({act.capitalize()})'
        
        ax.plot(grouped['P'], grouped['test_r2'],
                marker='s', linewidth=2.5, markersize=8, linestyle=linestyle,
                label=label, color=h2_colors.get(h2, 'gray'))

ax.set_xlabel('Polygenicity (P = # causal genes)', fontsize=13, fontweight='bold')
ax.set_ylabel('Test R²', fontsize=13, fontweight='bold')
ax.set_title(f'R² vs Polygenicity\nN={n_train_fixed}', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, ncol=1, handlelength=3)
ax.grid(alpha=0.3)
ax.set_xscale('log')
ax.tick_params(labelsize=11)

# Plot 2B (Right): R² vs Sample Size for α=1
ax = axes_epi[1]
subset_epistatic_n = results_combined[(results_combined['h2'] == h2_fixed) & 
                                       (results_combined['alpha'] == 1)]

print(f"Plot 2B (Epistatic, Sample Size) - Found {len(subset_epistatic_n)} rows for h²={h2_fixed}, α=1")

for P in sorted(subset_epistatic_n['P'].unique()):
    for act in ['relu', 'linear']:
        data = subset_epistatic_n[(subset_epistatic_n['P'] == P) & (subset_epistatic_n['activation'] == act)]
        grouped = data.groupby('n_train')['test_r2'].mean().reset_index()
        
        linestyle = '-' if act == 'relu' else '--'
        label = f'P={P} ({act.capitalize()})'
        
        ax.plot(grouped['n_train'], grouped['test_r2'],
                marker='^', linewidth=2.5, markersize=8, linestyle=linestyle,
                label=label, color=p_colors.get(P, 'gray'))

ax.set_xlabel('Training Sample Size (N)', fontsize=13, fontweight='bold')
ax.set_ylabel('Test R²', fontsize=13, fontweight='bold')
ax.set_title(f'R² vs Sample Size\nh²={h2_fixed}', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, ncol=2, handlelength=3)
ax.grid(alpha=0.3)
ax.tick_params(labelsize=11)

# Add super title for the figure
fig_epistatic.suptitle('PURE EPISTATIC (α=1): No Additive Effects', 
                       fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('../results/pure_synthetic_epistatic_grid.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Epistatic plot saved to: ../results/pure_synthetic_epistatic_grid.png")

Plot 1A (Additive, Polygenicity) - Found 8 rows for N=15000, α=0
Plot 1B (Additive, Sample Size) - Found 40 rows for h²=0.6, α=0

✓ Additive plot saved to: ../results/pure_synthetic_additive_grid.png

Plot 2A (Epistatic, Polygenicity) - Found 8 rows for N=15000, α=1
Plot 2B (Epistatic, Sample Size) - Found 40 rows for h²=0.6, α=1

✓ Epistatic plot saved to: ../results/pure_synthetic_epistatic_grid.png


In [ ]:
# # Plot 3: R² vs Training Sample Size (lines for each P × activation)
# ax = axes[2]

# # Filter for h²=0.5 (the only value in your data)
# h2_fixed = 0.6
# subset = results_combined[results_combined['h2'] == h2_fixed]

# print(f"Plot 3 - Found {len(subset)} rows for h²={h2_fixed}")

# for P in sorted(subset['P'].unique()):
#     for act in ['linear', 'relu']:
#         # Filter for this P and activation
#         data = subset[(subset['P'] == P) & (subset['activation'] == act)]
#         grouped = data.groupby('n_train')['test_r2'].mean().reset_index()
        
#         print(f"  P={P}, {act}: {len(grouped)} points")
        
#         linestyle = '-' if act == 'relu' else '--'
#         label = f'P={P} ({act.capitalize()})'
        
#         ax.plot(grouped['n_train'], grouped['test_r2'],
#                 marker='^', linewidth=2.5, markersize=8, linestyle=linestyle,
#                 label=label, color=p_colors.get(P, 'gray'))

# ax.set_xlabel('Training Sample Size (N)', fontsize=14, fontweight='bold')
# ax.set_ylabel('Test R²', fontsize=14, fontweight='bold')
# ax.set_title(f'R² vs Sample Size (h²={h2_fixed})', fontsize=14, fontweight='bold')
# ax.legend(fontsize=9, ncol=2)
# ax.grid(alpha=0.3)
# ax.tick_params(labelsize=12)

In [57]:
plt.tight_layout()
plt.savefig('../results/pure_synthetic_grid_study.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Plots saved to: ../results/pure_synthetic_grid_study.png")


✓ Plots saved to: ../results/pure_synthetic_grid_study.png


### R² vs Epistasis (Alpha) - Additional Plot

In [68]:
fig_epistasis, ax_epi = plt.subplots(1, 1, figsize=(10, 6))

# Fixed sample size for this plot
n_train_fixed = 15000
h2_fixed = 0.6

subset_alpha = results_combined[(results_combined['n_train'] == n_train_fixed) &
                                 (results_combined['h2'] == h2_fixed)]

print(f"\nEpistasis Plot - Found {len(subset_alpha)} rows for N={n_train_fixed}, h²={h2_fixed}")

# Plot for each P value with both activations
for P in sorted(subset_alpha['P'].unique()):
    # ReLU - solid line
    relu_data = subset_alpha[(subset_alpha['P'] == P) & (subset_alpha['activation'] == 'relu')]
    relu_grouped = relu_data.groupby('alpha')['test_r2'].mean().reset_index()
    ax_epi.plot(relu_grouped['alpha'], relu_grouped['test_r2'],
                marker='^', linewidth=2.5, markersize=8, linestyle='-',
                label=f'P={P} (ReLU)', color=p_colors.get(P, 'gray'))

    # Linear - dashed line
    linear_data = subset_alpha[(subset_alpha['P'] == P) & (subset_alpha['activation'] == 'linear')]
    linear_grouped = linear_data.groupby('alpha')['test_r2'].mean().reset_index()
    ax_epi.plot(linear_grouped['alpha'], linear_grouped['test_r2'],
                marker='^', linewidth=2.5, markersize=8, linestyle='--',
                label=f'P={P} (Linear)', color=p_colors.get(P, 'gray'), alpha=0.7)

    print(f"  P={P}: ReLU {len(relu_grouped)} points, Linear {len(linear_grouped)} points")

ax_epi.set_xlabel('Epistatic Proportion (α)', fontsize=14, fontweight='bold')
ax_epi.set_ylabel('Test R²', fontsize=14, fontweight='bold')
ax_epi.set_title(f'R² vs Epistasis (N={n_train_fixed}, h²={h2_fixed})', fontsize=18, fontweight='bold')
ax_epi.legend(title='Polygenicity & Activation', fontsize=10, title_fontsize=11, ncol=2)
ax_epi.grid(alpha=0.3)


ax_epi.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig('../results/pure_synthetic_epistasis_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Epistasis plot saved to: ../results/pure_synthetic_epistasis_plot.png")


Epistasis Plot - Found 24 rows for N=15000, h²=0.6
  P=1: ReLU 3 points, Linear 3 points
  P=10: ReLU 3 points, Linear 3 points
  P=50: ReLU 3 points, Linear 3 points
  P=100: ReLU 3 points, Linear 3 points

✓ Epistasis plot saved to: ../results/pure_synthetic_epistasis_plot.png


## Results Summary and Analysis

In [ ]:
plt.tight_layout()
plt.savefig('../results/pure_synthetic_grid_study.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Plots saved to: ../results/pure_synthetic_grid_study.png")

In [46]:
print("="*60)
print("SUMMARY STATISTICS")
print("="*60)

print("\n--- By Activation Type ---")
print(results_combined.groupby('activation')['test_r2'].agg(['mean', 'std', 'min', 'max']))

print("\n--- By Polygenicity (P) ---")
print(results_combined.groupby('P')['test_r2'].agg(['mean', 'std', 'min', 'max']))

print("\n--- By Heritability (h²) ---")
print(results_combined.groupby('h2')['test_r2'].agg(['mean', 'std', 'min', 'max']))

print("\n--- By Training Size (N) ---")
print(results_combined.groupby('n_train')['test_r2'].agg(['mean', 'std', 'min', 'max']))

print("\n" + "="*60)
print("ACTIVATION COMPARISON")
print("="*60)

comparison = results_combined.pivot_table(
    index=['P', 'h2', 'n_train'],
    columns='activation',
    values='test_r2'
)
comparison['advantage'] = comparison['relu'] - comparison['linear']

print("\nReLU Advantage (ReLU R² - Linear R²):")
print(comparison['advantage'].describe())

print("\n--- Scenarios where ReLU > Linear (top 10) ---")
top_relu = comparison.nlargest(10, 'advantage')[['relu', 'linear', 'advantage']]
print(top_relu)

print("\n--- Scenarios where Linear > ReLU (top 10) ---")
top_linear = comparison.nsmallest(10, 'advantage')[['relu', 'linear', 'advantage']]
print(top_linear)

SUMMARY STATISTICS

--- By Activation Type ---
                mean       std       min       max
activation                                        
linear     -0.176139  0.075576 -0.296491 -0.071071
relu       -0.215671  0.133993 -0.539999 -0.042249

--- By Polygenicity (P) ---
          mean       std       min       max
P                                           
1    -0.190803  0.106479 -0.407246 -0.042249
10   -0.205810  0.148227 -0.539999 -0.043139
100  -0.174927  0.081598 -0.306659 -0.042733
1000 -0.212079  0.103984 -0.364229 -0.045119

--- By Heritability (h²) ---
         mean       std       min       max
h2                                         
0.5 -0.195905  0.109226 -0.539999 -0.042249

--- By Training Size (N) ---
             mean       std       min       max
n_train                                        
1000    -0.162372  0.071026 -0.328110 -0.091879
2500    -0.213947  0.088961 -0.364229 -0.094947
5000    -0.327532  0.100860 -0.539999 -0.219619
10000   -0.209697 

In [47]:
print("="*60)
print("KEY FINDINGS")
print("="*60)

# Check if P effect is present
p_effect = results_combined[results_combined['h2'] == 0.6].groupby('P')['test_r2'].mean()
p_range = p_effect.max() - p_effect.min()

print(f"\n1. Polygenicity Effect (h²=0.6):")
print(f"   R² range across P values: {p_range:.3f}")
if p_range > 0.2:
    print("   ✓ STRONG P effect detected (>0.2 range)")
    print("   → Lower P (fewer SNPs) should have HIGHER R²")
elif p_range > 0.1:
    print("   ⚠ MODERATE P effect detected (0.1-0.2 range)")
else:
    print("   ✗ WEAK P effect (<0.1 range)")
    print("   → Standardization may still be affecting results")

print(f"\n   P=1: R² = {p_effect.get(1, 'N/A'):.3f}")
print(f"   P=1000: R² = {p_effect.get(1000, 'N/A'):.3f}")

# Check heritability effect
h2_effect = results_combined[results_combined['P'] == 100].groupby('h2')['test_r2'].mean()
print(f"\n2. Heritability Effect (P=100):")
for h2_val in [0.2, 0.6, 0.8]:
    print(f"   h²={h2_val}: R² = {h2_effect.get(h2_val, 'N/A'):.3f}")

# Check activation comparison
avg_advantage = comparison['advantage'].mean()
print(f"\n3. Activation Function Comparison:")
print(f"   Average ReLU advantage: {avg_advantage:.4f}")

if abs(avg_advantage) < 0.05:
    print("   ≈ No meaningful difference (as expected for pure additive)")
    print("   ✓ Both activations perform similarly on linear signal")
elif avg_advantage > 0.05:
    print("   ⚠ ReLU slightly better")
    print("   → May indicate non-linearity in data or training dynamics")
else:
    print("   ⚠ Linear slightly better")
    print("   → Expected for pure additive genetic architecture")

KEY FINDINGS

1. Polygenicity Effect (h²=0.6):
   R² range across P values: nan
   ✗ WEAK P effect (<0.1 range)
   → Standardization may still be affecting results


ValueError: Unknown format code 'f' for object of type 'str'